## Filter shapefile down by DA boundaries for the GTA

In [ ]:
import geopandas as gpd

# 1. Load the raw Dissemination Area shapefile
print("Loading Canada-wide DAs...")
da_boundaries = gpd.read_file("../data/01_raw/statcan_census/lda_000b21a_e.shp")

# 2. Define the exact CDUIDs that make up the Greater Toronto Area
gta_cduids = ["3520", "3521", "3519", "3518", "3524"]

# 3. Filter the GeoDataFrame using the .isin() method
print("Clipping boundaries to the GTA...")
gta_boundaries = da_boundaries[da_boundaries['DAUID'].str[:4].isin(gta_cduids)]

# 4. Convert to standard web mapping coordinates (essential for Kepler)
gta_boundaries = gta_boundaries.to_crs(epsg=4326)

print(f"Successfully isolated {len(gta_boundaries)} GTA neighborhoods.")


## Census Data columns

In [ ]:
import pandas as pd
import numpy as np
from tqdm.auto import tqdm

csv_path = "../data/01_raw/statcan_census/98-401-X2021006_English_CSV_data_Ontario.csv" 

# 1. Optimize the Read (The most important step)
# We only load the 3 columns we actually care about out of the dozens StatCan provides.
columns_we_need = ['ALT_GEO_CODE', 'CHARACTERISTIC_NAME', 'C1_COUNT_TOTAL']

# Count total lines for progress estimation
chunksize = 100_000
total_lines = sum(1 for _ in open(csv_path, encoding="cp1252")) - 1  # subtract header
total_chunks = (total_lines + chunksize - 1) // chunksize

# 2. Prepare an empty list to hold our filtered chunks
filtered_chunks = []

# 3. Stream the file in chunks of 100,000 rows at a time
print("Streaming and filtering the massive Census CSV...")
chunk_iterator = pd.read_csv(
    csv_path, 
    usecols=columns_we_need,
    dtype={"ALT_GEO_CODE": str}, # Preserve leading zeros on the DAUID!
    chunksize=chunksize,
    low_memory=False,
    encoding="cp1252"
)

# 4. Process each chunk individually
for chunk in tqdm(chunk_iterator, total=total_chunks, desc="Filtering chunks"):
    # Filter the current chunk for our two target metrics
    pop_mask = chunk['CHARACTERISTIC_NAME'] == 'Population, 2021'
    income_mask = chunk['CHARACTERISTIC_NAME'].str.contains('Median total income in 2020 among recipients', na=False, regex=False)
    
    # Keep only the matching rows and append to our list
    kept_rows = chunk[pop_mask | income_mask]
    filtered_chunks.append(kept_rows)

# 5. Combine all the tiny filtered chunks back into one standard DataFrame
print("Combining filtered chunks...")
master_filtered_df = pd.concat(filtered_chunks, ignore_index=True)

# 6. Apply the Pivot (from long format to wide format)
print("Pivoting the final dataset...")
master_filtered_df['Metric'] = np.where(
    master_filtered_df['CHARACTERISTIC_NAME'] == 'Population, 2021', 
    'Population', 
    'Median_Income'
)

clean_demographics = master_filtered_df.pivot_table(
    index='ALT_GEO_CODE',
    columns='Metric',
    values='C1_COUNT_TOTAL',
    aggfunc='first'
).reset_index()

# Clean up the suppressed data (turn 'x' or '...' into clean NaNs)
clean_demographics['Population'] = pd.to_numeric(clean_demographics['Population'], errors='coerce')
clean_demographics['Median_Income'] = pd.to_numeric(clean_demographics['Median_Income'], errors='coerce')

print("\nSuccess! Final memory-safe demographic table created.")
print(clean_demographics.head())